# ECE 3715 Mini-Project 2

**Seed: 611441029**  
**Steps 1–2: Christian Ruelas**

Steps 1 and 2 establish the continuous distributions used throughout the project and then rebuild
the exponential and Rayleigh generators directly from uniform random variables.

The same fixed seed is used every run, with an independent generator assigned to each project step.


In [ ]:
from time import perf_counter
START = perf_counter()

from pathlib import Path
import os
import sys
import numpy as np

# Keep Matplotlib's cache inside the project so it is always writable.
os.environ.setdefault("MPLCONFIGDIR", str(Path(".mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display, Markdown

from mp2lib import (
    inv_cdf_exponential,
    inv_cdf_rayleigh,
    sample_by_inversion,
    ks_statistic,
    q_function,
)

SEED = 611441029
rng = np.random.default_rng(SEED)

rng1 = rng.spawn(1)[0]
rng2 = rng.spawn(1)[0]
rng3 = rng.spawn(1)[0]
rng4 = rng.spawn(1)[0]
rng5 = rng.spawn(1)[0]
rng6 = rng.spawn(1)[0]
rng7 = rng.spawn(1)[0]
rng8 = rng.spawn(1)[0]
rng_test = rng.spawn(1)[0]

print(f"Seed: {SEED}")
print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, Matplotlib {matplotlib.__version__}")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 160,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.spines.top": False,
})

def table(headers, rows):
    """Display a readable Markdown table."""
    def fmt(x):
        if isinstance(x, (float, np.floating)):
            return f"{x:.7g}"
        return str(x).replace("|", r"\|")

    body = "\n".join(
        "| " + " | ".join(fmt(x) for x in row) + " |"
        for row in rows
    )
    display(Markdown(
        "| " + " | ".join(headers) + " |\n"
        + "|" + "|".join(["---"] * len(headers)) + "|\n"
        + body
    ))

def finish(fig, name):
    """Save a figure and display it exactly once."""
    fig.savefig(FIGDIR / f"{name}.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)


### Library function tests

The inverse-CDF functions are checked at hand-calculable values. At $u=0$, both generators must
return 0. At $u=0.5$, the exponential result is $\ln 2$ and the Rayleigh result is
$\sqrt{2\ln 2}$. Also, $Q(0)=0.5$.


In [ ]:
u = np.array([0.0, 0.5])

assert np.allclose(inv_cdf_exponential(u, 1.0), [0.0, np.log(2.0)])
assert np.allclose(inv_cdf_rayleigh(u, 1.0), [0.0, np.sqrt(2.0 * np.log(2.0))])

draw = sample_by_inversion(inv_cdf_exponential, 5, rng_test, lam=1.0)
assert draw.shape == (5,)
assert np.all(draw >= 0)

assert np.isclose(q_function(0.0), 0.5)

print("All Step 1–2 library function tests passed.")


## Step 1 — From CDF to density

Four distributions are used: Uniform on $[0,1]$, Exponential with rate $\lambda=1$,
standard Gaussian, and Rayleigh with scale $\sigma=1$.

**Uniform $U(0,1)$**

&emsp;&emsp;CDF: $F_X(x)=0$ for $x<0$, $F_X(x)=x$ for $0\le x\le1$, and $F_X(x)=1$ for $x>1$.

&emsp;&emsp;PDF: $f_X(x)=1$ for $0\le x\le1$ and $0$ otherwise.

&emsp;&emsp;Support: $0\le x\le1$.

**Exponential, $\lambda=1$**

&emsp;&emsp;CDF: $F_X(x)=1-e^{-x}$ for $x\ge0$ and $0$ for $x<0$.

&emsp;&emsp;PDF: $f_X(x)=e^{-x}$ for $x\ge0$ and $0$ for $x<0$.

&emsp;&emsp;Support: $x\ge0$.

**Standard Gaussian**

&emsp;&emsp;CDF: $F_X(x)=\Phi(x)$.

&emsp;&emsp;PDF: $f_X(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}$.

&emsp;&emsp;Support: $-\infty<x<\infty$.

**Rayleigh, $\sigma=1$**

&emsp;&emsp;CDF: $F_X(x)=1-e^{-x^2/2}$ for $x\ge0$ and $0$ for $x<0$.

&emsp;&emsp;PDF: $f_X(x)=xe^{-x^2/2}$ for $x\ge0$ and $0$ for $x<0$.

&emsp;&emsp;Support: $x\ge0$.


In [ ]:
N1 = 100_000
lam = 1.0
sigma = 1.0

# Fixed draw order for reproducibility.
exponential_samples = rng1.exponential(scale=1.0 / lam, size=N1)
gaussian_samples = rng1.normal(loc=0.0, scale=1.0, size=N1)
rayleigh_samples = rng1.rayleigh(scale=sigma, size=N1)
uniform_samples = rng1.uniform(0.0, 1.0, size=N1)

print(f"Generated {N1:,} samples for each Step 1 distribution.")


In [ ]:
def uniform_cdf(x):
    x = np.asarray(x)
    return np.where(x < 0, 0.0, np.where(x <= 1, x, 1.0))

def uniform_pdf(x):
    x = np.asarray(x)
    return np.where((x >= 0) & (x <= 1), 1.0, 0.0)

def exponential_cdf(x):
    x = np.asarray(x)
    return np.where(x < 0, 0.0, 1.0 - np.exp(-x))

def exponential_pdf(x):
    x = np.asarray(x)
    return np.where(x < 0, 0.0, np.exp(-x))

def gaussian_cdf(x):
    return norm.cdf(x)

def gaussian_pdf(x):
    return norm.pdf(x)

def rayleigh_cdf(x):
    x = np.asarray(x)
    return np.where(x < 0, 0.0, 1.0 - np.exp(-(x**2) / 2.0))

def rayleigh_pdf(x):
    x = np.asarray(x)
    return np.where(x < 0, 0.0, x * np.exp(-(x**2) / 2.0))


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))

plots = [
    ("Uniform", uniform_samples, uniform_cdf, uniform_pdf, np.linspace(0, 1, 500)),
    ("Exponential", exponential_samples, exponential_cdf, exponential_pdf, np.linspace(0, 8, 500)),
    ("Gaussian", gaussian_samples, gaussian_cdf, gaussian_pdf, np.linspace(-4, 4, 500)),
    ("Rayleigh", rayleigh_samples, rayleigh_cdf, rayleigh_pdf, np.linspace(0, 5, 500)),
]

for j, (name, samples, cdf, pdf, xgrid) in enumerate(plots):
    xs = np.sort(samples)
    ecdf = np.arange(1, N1 + 1) / N1

    axes[0, j].plot(xs, ecdf, label="Empirical CDF")
    axes[0, j].plot(xgrid, cdf(xgrid), "--", label="Analytic CDF")
    axes[0, j].set_title(name)
    axes[0, j].set_xlabel("x")
    axes[0, j].set_ylabel("CDF")
    axes[0, j].legend()

    axes[1, j].hist(samples, bins=80, density=True, alpha=0.60, label="Histogram")
    axes[1, j].plot(xgrid, pdf(xgrid), linewidth=2, label="Analytic PDF")
    axes[1, j].set_xlim(xgrid[0], xgrid[-1])
    axes[1, j].set_xlabel("x")
    axes[1, j].set_ylabel("Density")
    axes[1, j].legend()

fig.suptitle("Figure 1 — Empirical CDFs and densities")
fig.tight_layout()
finish(fig, "figure1")


### Figure 1 interpretation

The empirical CDFs track the analytic CDFs closely, and the normalized histograms follow the
corresponding analytic densities. The quantitative goodness-of-fit check is the KS statistic below;
the curves are not accepted based on appearance alone.


In [ ]:
ks_critical_1 = 1.36 / np.sqrt(N1)

ks_rows = [
    ["Uniform", ks_statistic(uniform_samples, uniform_cdf), ks_critical_1],
    ["Exponential", ks_statistic(exponential_samples, exponential_cdf), ks_critical_1],
    ["Gaussian", ks_statistic(gaussian_samples, gaussian_cdf), ks_critical_1],
    ["Rayleigh", ks_statistic(rayleigh_samples, rayleigh_cdf), ks_critical_1],
]

table(["Distribution", "KS statistic", "95% critical value"], ks_rows)

print("All Step 1 KS statistics below critical value:",
      all(row[1] < row[2] for row in ks_rows))


For the Exponential distribution with $\lambda=1$,

&emsp;&emsp;$E[X]=1/\lambda=1,$

&emsp;&emsp;$\operatorname{median}(X)=\ln(2)/\lambda=0.6931,$

&emsp;&emsp;$\operatorname{Var}(X)=1/\lambda^2=1.$

For the Rayleigh distribution with $\sigma=1$,

&emsp;&emsp;$E[X]=\sigma\sqrt{\pi/2}=1.2533,$

&emsp;&emsp;$\operatorname{median}(X)=\sigma\sqrt{2\ln2}=1.1774,$

&emsp;&emsp;$\operatorname{Var}(X)=\frac{4-\pi}{2}\sigma^2=0.4292.$


In [ ]:
moment_rows = [
    ["Exponential mean", np.mean(exponential_samples), 1.0],
    ["Exponential median", np.median(exponential_samples), np.log(2.0)],
    ["Exponential variance", np.var(exponential_samples), 1.0],
    ["Rayleigh mean", np.mean(rayleigh_samples), np.sqrt(np.pi / 2.0)],
    ["Rayleigh median", np.median(rayleigh_samples), np.sqrt(2.0 * np.log(2.0))],
    ["Rayleigh variance", np.var(rayleigh_samples), (4.0 - np.pi) / 2.0],
]

table(["Quantity", "Sample", "Closed form"], moment_rows)


The exponential mean is about 1.44 times its median because the distribution is right-skewed:
its long right tail pulls the mean upward. The Gaussian density is symmetric about zero, so its
mean and median coincide.


## Step 2 — Build your own generator by inversion

Let $U\sim U(0,1)$ and let $F$ be a continuous, strictly increasing CDF. If
$X=F^{-1}(U)$, then

&emsp;&emsp;$P(X\le x)=P(F^{-1}(U)\le x)=P(U\le F(x))=F(x).$

Therefore $F^{-1}(U)$ has CDF $F$.

For the Exponential distribution,

&emsp;&emsp;$u=1-e^{-\lambda x}$

&emsp;&emsp;$1-u=e^{-\lambda x}$

&emsp;&emsp;$\ln(1-u)=-\lambda x$

&emsp;&emsp;$F^{-1}(u)=-\frac{\ln(1-u)}{\lambda}.$

For the Rayleigh distribution,

&emsp;&emsp;$u=1-e^{-x^2/(2\sigma^2)}$

&emsp;&emsp;$\ln(1-u)=-\frac{x^2}{2\sigma^2}$

&emsp;&emsp;$x^2=-2\sigma^2\ln(1-u)$

&emsp;&emsp;$F^{-1}(u)=\sigma\sqrt{-2\ln(1-u)}.$


In [ ]:
N2 = 100_000

exp_inv = sample_by_inversion(
    inv_cdf_exponential, N2, rng2, lam=1.0
)

ray_inv = sample_by_inversion(
    inv_cdf_rayleigh, N2, rng2, sigma=1.0
)

ks_exp_inv = ks_statistic(exp_inv, exponential_cdf)
ks_ray_inv = ks_statistic(ray_inv, rayleigh_cdf)
ks_critical_2 = 1.36 / np.sqrt(N2)

table(
    ["Distribution", "KS statistic", "95% critical value", "Pass"],
    [
        ["Exponential", ks_exp_inv, ks_critical_2, ks_exp_inv < ks_critical_2],
        ["Rayleigh", ks_ray_inv, ks_critical_2, ks_ray_inv < ks_critical_2],
    ],
)


In [ ]:
# Comparison against NumPy library generators.
exp_library = rng2.exponential(scale=1.0, size=N2)
ray_library = rng2.rayleigh(scale=1.0, size=N2)

table(
    ["Distribution", "Inversion mean", "Library mean", "Theory"],
    [
        ["Exponential", np.mean(exp_inv), np.mean(exp_library), 1.0],
        ["Rayleigh", np.mean(ray_inv), np.mean(ray_library), np.sqrt(np.pi / 2.0)],
    ],
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

x_exp = np.linspace(0, 8, 500)
xs = np.sort(exp_inv)
ecdf = np.arange(1, N2 + 1) / N2
axes[0].plot(xs, ecdf, label=f"Inversion, KS={ks_exp_inv:.4f}")
axes[0].plot(x_exp, exponential_cdf(x_exp), "--", label="Analytic CDF")
axes[0].set_xlim(0, 8)
axes[0].set_title("Exponential")
axes[0].set_xlabel("x")
axes[0].set_ylabel("CDF")
axes[0].legend()

x_ray = np.linspace(0, 5, 500)
xs = np.sort(ray_inv)
ecdf = np.arange(1, N2 + 1) / N2
axes[1].plot(xs, ecdf, label=f"Inversion, KS={ks_ray_inv:.4f}")
axes[1].plot(x_ray, rayleigh_cdf(x_ray), "--", label="Analytic CDF")
axes[1].set_xlim(0, 5)
axes[1].set_title("Rayleigh")
axes[1].set_xlabel("x")
axes[1].set_ylabel("CDF")
axes[1].legend()

fig.suptitle("Figure 2 — Inverse-transform generators")
fig.tight_layout()
finish(fig, "figure2")


### Figure 2 interpretation

Both inverse-transform empirical CDFs follow the analytic CDFs, and both KS statistics are below
the $N=10^5$ 95% critical value of about 0.0043. The library-generator means also agree with the
inverse-transform means and their closed-form expectations.

`rng.random()` samples from $[0,1)$, so $U=0$ is possible but $U=1$ is excluded. With the
implemented $-\ln(1-U)$ formulas, $U=0$ maps safely to 0; `np.log1p(-u)` also evaluates
$\ln(1-u)$ accurately near zero.

The exponential inverse may also be written $-\ln(U)/\lambda$ because $1-U$ is uniform whenever
$U$ is uniform. This notebook uses $-\ln(1-U)/\lambda$ because it follows directly from solving
the CDF for $x$.


In [ ]:
print(f"Runtime through Step 2: {perf_counter() - START:.2f} s")


## Step 3

Continue the shared notebook here.
